# Explore here

In [17]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt 
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import chi2, SelectKBest
from sklearn.model_selection import train_test_split


#Imported Necessary Libraries // Librerias necesarias para el proyecto. 

In [18]:
# Load Database / Cargar base de datos.
df = pd.read_csv('/workspaces/Tutorial-Construye-un-modelo-de-regresion-lineal-usando-pandas-y-python/data/raw/demographic_health_data.csv')
total_data = df 
total_data.head()

,fips,TOT_POP,0-9,0-9 y/o % of total pop,19-Oct,10-19 y/o % of total pop,20-29,20-29 y/o % of total pop,30-39,30-39 y/o % of total pop,...,COPD_number,diabetes_prevalence,diabetes_Lower 95% CI,diabetes_Upper 95% CI,diabetes_number,CKD_prevalence,CKD_Lower 95% CI,CKD_Upper 95% CI,CKD_number,Urban_rural_code
0,1001,55601,6787,12.206615,7637,13.735364,6878,12.370281,7089,12.749771,...,3644,12.9,11.9,13.8,5462,3.1,2.9,3.3,1326,3
1,1003,218022,24757,11.355276,26913,12.344167,23579,10.814964,25213,11.564429,...,14692,12.0,11.0,13.1,20520,3.2,3.0,3.5,5479,4
2,1005,24881,2732,10.980266,2960,11.896628,3268,13.134520,3201,12.865239,...,2373,19.7,18.6,20.6,3870,4.5,4.2,4.8,887,6
3,1007,22400,2456,10.964286,2596,11.589286,3029,13.522321,3113,13.897321,...,1789,14.1,13.2,14.9,2511,3.3,3.1,3.6,595,2
4,1009,57840,7095,12.266598,7570,13.087828,6742,11.656293,6884,11.901798,...,4661,13.5,12.6,14.5,6017,3.4,3.2,3.7,1507,2


In [19]:
print(total_data.columns.tolist())
# Print columns to select the target variable for the health study due to socioeconomics issues. / Imprimir las columnas para seleccionar la variable
# objetivo del estudio de salud en cuanto a aspectos socioeconomicos. 

#The chosen variable it is "anycondition_prevalence" (now used as target), because it resumes all possible health problems of the population / La 
# variable elegida es "anycondition_prevalence" (ahora usada como objetivo), porque resume todos los posibles problemas de salud de la población.

['fips', 'TOT_POP', '0-9', '0-9 y/o % of total pop', '19-Oct', '10-19 y/o % of total pop', '20-29', '20-29 y/o % of total pop', '30-39', '30-39 y/o % of total pop', '40-49', '40-49 y/o % of total pop', '50-59', '50-59 y/o % of total pop', '60-69', '60-69 y/o % of total pop', '70-79', '70-79 y/o % of total pop', '80+', '80+ y/o % of total pop', 'White-alone pop', '% White-alone', 'Black-alone pop', '% Black-alone', 'Native American/American Indian-alone pop', '% NA/AI-alone', 'Asian-alone pop', '% Asian-alone', 'Hawaiian/Pacific Islander-alone pop', '% Hawaiian/PI-alone', 'Two or more races pop', '% Two or more races', 'POP_ESTIMATE_2018', 'N_POP_CHG_2018', 'GQ_ESTIMATES_2018', 'R_birth_2018', 'R_death_2018', 'R_NATURAL_INC_2018', 'R_INTERNATIONAL_MIG_2018', 'R_DOMESTIC_MIG_2018', 'R_NET_MIG_2018', 'Less than a high school diploma 2014-18', 'High school diploma only 2014-18', "Some college or associate's degree 2014-18", "Bachelor's degree or higher 2014-18", 'Percent of adults with les

In [20]:
# Database features / Características de la base de datos
print(total_data.shape)
print(total_data.describe())
print(total_data.info())

(3140, 108)
               fips       TOT_POP           0-9  0-9 y/o % of total pop  \
count   3140.000000  3.140000e+03  3.140000e+03             3140.000000   
mean   30401.640764  1.041894e+05  1.274030e+04               11.871051   
std    15150.559265  3.335834e+05  4.180730e+04                2.124081   
min     1001.000000  8.800000e+01  0.000000e+00                0.000000   
25%    18180.500000  1.096325e+04  1.280500e+03               10.594639   
50%    29178.000000  2.580050e+04  3.057000e+03               11.802727   
75%    45081.500000  6.791300e+04  8.097000e+03               12.951840   
max    56045.000000  1.010552e+07  1.208253e+06               25.460677   

             19-Oct  10-19 y/o % of total pop         20-29  \
count  3.140000e+03               3140.000000  3.140000e+03   
mean   1.336798e+04                 12.694609  1.446933e+04   
std    4.228439e+04                  1.815044  4.957773e+04   
min    0.000000e+00                  0.000000  0.000000e+00 

In [21]:
#Eliminate duplicates / Eliminar duplicados. 
total_data[total_data.duplicated(keep=False)]
total_data = total_data.drop_duplicates()
total_data.shape

(3140, 108)

In [22]:
#Check for null values / Verificar valores nulos.
total_data.isnull().sum()

fips                      0
TOT_POP                   0
0-9                       0
0-9 y/o % of total pop    0
19-Oct                    0
                         ..
CKD_prevalence            0
CKD_Lower 95% CI          0
CKD_Upper 95% CI          0
CKD_number                0
Urban_rural_code          0
Length: 108, dtype: int64

In [ ]:
#Determinate irrelevant features / Determinar características irrelevantes.
# As we chose "anycondition_prevalence" as target variable, we will eliminate the features and variables linked to any ohter case. 
# Focusing on the socioconomis aspects. Also duplicates of information stored in different columns and ways. /
# Como elegimos "anycondition_prevalence" como variable objetivo, eliminaremos las características y variables vinculadas a cualquier otro caso.
# Enfocándonos en los aspectos socioconomicos. También duplicados de información almacenada en diferentes columnas y formas.

total_data = total_data.drop([ 'fips', 'STATE_FIPS', 'CNTY_FIPS'], axis=1) 

# No predictive features / Características no predictivas.

total_data = total_data.drop([
    '0-9', '19-Oct', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80+',
    'White-alone pop', 'Black-alone pop',
    'Native American/American Indian-alone pop',
    'Asian-alone pop', 'Hawaiian/Pacific Islander-alone pop',
    'Two or more races pop',
    'Less than a high school diploma 2014-18',
    'High school diploma only 2014-18',
    "Some college or associate's degree 2014-18",
    "Bachelor's degree or higher 2014-18",
    'POVALL_2018',
    'Civilian_labor_force_2018', 'Employed_2018', 'Unemployed_2018',
    'Population Aged 60+', 'Total Population', 'POP_ESTIMATE_2018',], axis=1) 

# % version replaced features / Características reemplazadas por porcentajes.

total_data = total_data.drop([
    'Total Active Patient Care Physicians per 100000 Population 2018 (AAMC)',
    'Active Patient Care Primary Care Physicians per 100000 Population 2018 (AAMC)',
    'Active General Surgeons per 100000 Population 2018 (AAMC)',
    'Active Patient Care General Surgeons per 100000 Population 2018 (AAMC)',
    'MEDHHINC_2018',          
    'PCTPOV017_2018',          
    'PCTPOV517_2018',          
], axis=1)

#Almos perfect duplicates / Casi duplicados.

total_data = total_data.drop([
    'R_NATURAL_INC_2018',                    # = R_birth - R_death
    'R_NET_MIG_2018',                        # = R_DOM + R_INTL
    'N_POP_CHG_2018',
    'GQ_ESTIMATES_2018',
    'CI90LBINC_2018', 'CI90UBINC_2018',      # intervalos de confianza del ingreso
    'Med_HH_Income_Percent_of_State_Total_2018',
    'county_pop2018_18 and older',
], axis=1)

# Derived by another variables or redundant metrics./ Metricas derivadas de otras variables o redundantes.

total_data = total_data.drop([
    'R_birth_2018',
    'R_DOMESTIC_MIG_2018',
    'R_INTERNATIONAL_MIG_2018',
], axis=1)

# Irrelevant for the target variable / Irrelevantes para la variable objetivo.

total_data = total_data.drop([
    'R_death_2018',
    'anycondition_Lower 95% CI', 'anycondition_Upper 95% CI', 'anycondition_number',
    'Obesity_prevalence', 'Obesity_Lower 95% CI', 'Obesity_Upper 95% CI', 'Obesity_number',
    'Heart disease_prevalence', 'Heart disease_Lower 95% CI',
    'Heart disease_Upper 95% CI', 'Heart disease_number',
    'COPD_prevalence', 'COPD_Lower 95% CI', 'COPD_Upper 95% CI', 'COPD_number',
    'diabetes_prevalence', 'diabetes_Lower 95% CI',
    'diabetes_Upper 95% CI', 'diabetes_number',
    'CKD_prevalence', 'CKD_Lower 95% CI', 'CKD_Upper 95% CI', 'CKD_number',
], axis=1)

# Cronic conditions correlated to the target variable, but not useful for the model. / 
# Condiciones crónicas correlacionadas con la variable objetivo, pero no útiles para el modelo.

total_data.info()
print(f"\nDimensiones: {total_data.shape}")   
print(total_data.head())





<class 'pandas.DataFrame'>
RangeIndex: 3140 entries, 0 to 3139
Data columns (total 37 columns):
 #   Column                                                                   Non-Null Count  Dtype  
---  ------                                                                   --------------  -----  
 0   TOT_POP                                                                  3140 non-null   int64  
 1   0-9 y/o % of total pop                                                   3140 non-null   float64
 2   10-19 y/o % of total pop                                                 3140 non-null   float64
 3   20-29 y/o % of total pop                                                 3140 non-null   float64
 4   30-39 y/o % of total pop                                                 3140 non-null   float64
 5   40-49 y/o % of total pop                                                 3140 non-null   float64
 6   50-59 y/o % of total pop                                                 3140 non-n